In [ ]:
import requests
import pandas as pd
import os

FMP_API_KEY = "JcmfjzcBwo5HGSiM5Yib7ylfG2PmSNzc" 

def fetch_fmp_hourly_data(ticker, api_key, interval="1hour", limit=1000, save_dir="FMP_Hourly"):
    """
    Fetch hourly stock data using FMP intraday endpoint (paid plans only).

    Parameters:
        ticker (str): Stock symbol (e.g., "AAPL")
        api_key (str): FMP API Key
        interval (str): "1min", "5min", "15min", "30min", "1hour"
        limit (int): Number of data points to fetch (max ~4000+ for paid plans)
        save_dir (str): Directory to save CSV file
    """
    print(f"Fetching {interval} intraday data for {ticker}...")

    url = f"https://financialmodelingprep.com/api/v3/historical-chart/{interval}/{ticker}?limit={limit}&apikey={api_key}"
    response = requests.get(url)

    if response.status_code != 200:
        raise Exception(f"Failed to fetch data: {response.status_code} - {response.text}")

    data = response.json()
    if not data:
        raise Exception("No data returned. Check API key, subscription level, or ticker.")

    df = pd.DataFrame(data)
    df.rename(columns={
        "date": "Datetime",
        "open": "Open_Prices",
        "high": "High_Prices",
        "low": "Low_Prices",
        "close": "Close_Prices",
        "volume": "Volume"
    }, inplace=True)
    df["Datetime"] = pd.to_datetime(df["Datetime"])
    df = df.sort_values("Datetime")

    # Save to CSV
    os.makedirs(save_dir, exist_ok=True)
    filepath = os.path.join(save_dir, f"{ticker.upper()}_{interval}_data.csv")
    df.to_csv(filepath, index=False)

    print(f"✅ Hourly data for {ticker} saved to {filepath}")
    return df


In [13]:
# Run this cell to fetch hourly data for any ticker
ticker = input("Enter stock ticker (e.g., AAPL): ").strip().upper()
df_hourly = fetch_fmp_hourly_data(ticker, FMP_API_KEY, interval="1hour", limit=2000)

# View sample data
df_hourly.head()


Fetching 1hour intraday data for AAPL...
✅ Hourly data for AAPL saved to FMP_Hourly\AAPL_1hour_data.csv


,Datetime,Open_Prices,Low_Prices,High_Prices,Close_Prices,Volume
453,2024-12-30 09:30:00,252.23,250.75,253.00,250.92,7399637
452,2024-12-30 10:30:00,250.93,250.89,252.29,251.90,3565275
451,2024-12-30 11:30:00,251.89,251.81,253.10,253.05,2716490
450,2024-12-30 12:30:00,253.06,252.50,253.47,252.82,2153979
449,2024-12-30 13:30:00,252.80,252.59,253.29,252.94,2351653
